# Notebook 02 Part 2: `select()`, `filter()`, and `withColumn()`

In **Notebook 02 Part 1**, we learned how to create `SparkSession`, create a DataFrame, display data, check schema, check columns, and count rows.

## Topics Covered

```text
1. Windows local setup cell
2. Create SparkSession
3. Create sample DataFrame
4. select()
5. filter()
6. Multiple filter conditions
7. withColumn()
8. col()
9. lit()
10. Derived columns
11. Practice questions
```


## 1. Windows Local Setup Cell

Run this cell before creating `SparkSession`.

This cell helps avoid the common Windows error:

```text
Timed out while waiting for the Python worker to connect back
```


In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"


## 2. Import SparkSession

`SparkSession` is the starting point of PySpark.


In [14]:
from pyspark.sql import SparkSession

## 3. Create SparkSession

We will use the Windows-safe SparkSession configuration.

| Code | Meaning |
|---|---|
| `.appName("pyspark")` | Name of Spark application |
| `.master("local[1]")` | Run locally using 1 worker thread |
| `spark.python.worker.reuse = false` | Avoid Python worker reuse issue |
| `spark.driver.host = 127.0.0.1` | Use localhost |
| `spark.driver.bindAddress = 127.0.0.1` | Bind Spark driver to localhost |


In [15]:
spark = SparkSession.builder \
    .appName("pyspark") \
    .master("local[1]") \
    .config("spark.python.worker.reuse", "false") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

## 4. Create Sample DataFrame

We will use a small sales/order dataset.

Columns:

```text
order_id, customer_name, city, amount
```


In [16]:
data = [
    (1, "Amit", "Delhi", 500),
    (2, "Priya", "Mumbai", 700),
    (3, "Rahul", "Delhi", 300),
    (4, "Sneha", "Pune", 900),
    (5, "Neha", "Mumbai", 400),
    (6, "Karan", "Delhi", 1000)
]

columns = ["order_id", "customer_name", "city", "amount"]

df = spark.createDataFrame(data, columns)

df.show()

+--------+-------------+------+------+
|order_id|customer_name|  city|amount|
+--------+-------------+------+------+
|       1|         Amit| Delhi|   500|
|       2|        Priya|Mumbai|   700|
|       3|        Rahul| Delhi|   300|
|       4|        Sneha|  Pune|   900|
|       5|         Neha|Mumbai|   400|
|       6|        Karan| Delhi|  1000|
+--------+-------------+------+------+



## 5. `select()` Command

`select()` is used to choose columns from a DataFrame.

Simple meaning:

```text
select() = choose columns
```


In [17]:
df.select("customer_name").show()

+-------------+
|customer_name|
+-------------+
|         Amit|
|        Priya|
|        Rahul|
|        Sneha|
|         Neha|
|        Karan|
+-------------+



## 6. Select Multiple Columns

We can pass multiple column names inside `select()`.


In [18]:
df.select("customer_name", "city").show()

+-------------+------+
|customer_name|  city|
+-------------+------+
|         Amit| Delhi|
|        Priya|Mumbai|
|        Rahul| Delhi|
|        Sneha|  Pune|
|         Neha|Mumbai|
|        Karan| Delhi|
+-------------+------+



## 7. Select Columns in Different Order

The output follows the order we give inside `select()`.


In [19]:
df.select("amount", "city", "customer_name").show()

+------+------+-------------+
|amount|  city|customer_name|
+------+------+-------------+
|   500| Delhi|         Amit|
|   700|Mumbai|        Priya|
|   300| Delhi|        Rahul|
|   900|  Pune|        Sneha|
|   400|Mumbai|         Neha|
|  1000| Delhi|        Karan|
+------+------+-------------+



## 8. `filter()` Command

`filter()` is used to select rows based on a condition.

Simple meaning:

```text
filter() = choose rows
```


In [20]:
df.filter(df.city == "Delhi").show()

+--------+-------------+-----+------+
|order_id|customer_name| city|amount|
+--------+-------------+-----+------+
|       1|         Amit|Delhi|   500|
|       3|        Rahul|Delhi|   300|
|       6|        Karan|Delhi|  1000|
+--------+-------------+-----+------+



## 9. Filter Amount Greater Than 500

We can use comparison operators.

| Operator | Meaning |
|---|---|
| `==` | Equal to |
| `!=` | Not equal to |
| `>` | Greater than |
| `<` | Less than |
| `>=` | Greater than or equal to |
| `<=` | Less than or equal to |


In [9]:
df.filter(df.amount > 500).show()

+--------+-------------+------+------+
|order_id|customer_name|  city|amount|
+--------+-------------+------+------+
|       2|        Priya|Mumbai|   700|
|       4|        Sneha|  Pune|   900|
|       6|        Karan| Delhi|  1000|
+--------+-------------+------+------+



## 10. Filter Amount Less Than or Equal to 500

In [10]:
df.filter(df.amount <= 500).show()

+--------+-------------+------+------+
|order_id|customer_name|  city|amount|
+--------+-------------+------+------+
|       1|         Amit| Delhi|   500|
|       3|        Rahul| Delhi|   300|
|       5|         Neha|Mumbai|   400|
+--------+-------------+------+------+



## 11. Select and Filter Together

Example:

```text
First filter rows where amount > 500
Then select only customer_name and amount
```


In [11]:
df.filter(df.amount > 500).select("customer_name", "amount").show()

+-------------+------+
|customer_name|amount|
+-------------+------+
|        Priya|   700|
|        Sneha|   900|
|        Karan|  1000|
+-------------+------+



## 12. Multiple Conditions

For multiple conditions, we use:

| Symbol | Meaning |
|---|---|
| `&` | AND |
| `|` | OR |

Important: each condition should be inside brackets.


### AND Condition

Example:

```text
city is Mumbai AND amount is greater than 500
```


In [12]:
df.filter((df.city == "Mumbai") & (df.amount > 500)).show()

+--------+-------------+------+------+
|order_id|customer_name|  city|amount|
+--------+-------------+------+------+
|       2|        Priya|Mumbai|   700|
+--------+-------------+------+------+



### OR Condition

Example:

```text
city is Delhi OR amount is greater than 800
```


In [13]:
df.filter((df.city == "Delhi") | (df.amount > 800)).show()

+--------+-------------+-----+------+
|order_id|customer_name| city|amount|
+--------+-------------+-----+------+
|       1|         Amit|Delhi|   500|
|       3|        Rahul|Delhi|   300|
|       4|        Sneha| Pune|   900|
|       6|        Karan|Delhi|  1000|
+--------+-------------+-----+------+



## 13. Common Mistake in Multiple Conditions

Wrong:

```python
df.filter(df.city == "Delhi" & df.amount > 300)
```

Correct:

```python
df.filter((df.city == "Delhi") & (df.amount > 300))
```


## 14. Import PySpark Functions

Now we will use two important PySpark functions:

| Function | Meaning |
|---|---|
| `col()` | Refers to a DataFrame column |
| `lit()` | Adds a fixed/literal value |


In [ ]:
from pyspark.sql.functions import col, lit

## 15. `withColumn()` Command

`withColumn()` is used to:

```text
1. Add a new column
2. Update an existing column
```

Syntax:

```python
df.withColumn("new_column_name", expression)
```


## 16. Add a Fixed Column using `lit()`

Example:

```text
Add a new column country with fixed value India
```


In [ ]:
df_country = df.withColumn("country", lit("India"))

df_country.show()

## 17. Add a Derived Column using `col()`

A derived column is created using existing columns.

Example:

```text
amount_after_tax = amount * 1.10
```

Here we assume 10% tax.


In [ ]:
df_tax = df.withColumn("amount_after_tax", col("amount") * 1.10)

df_tax.show()

## 18. Add Discount Column

Example:

```text
discount_amount = amount * 0.10
```


In [ ]:
df_discount = df.withColumn("discount_amount", col("amount") * 0.10)

df_discount.show()

## 19. Add Final Amount Column

Now we will create:

```text
final_amount = amount - discount_amount
```

We first create `discount_amount`, then use it to create `final_amount`.


``` bash
df_new = df.withColumn("new_column_1", expression_1) \
    .withColumn("new_column_2", expression_2) \
    .withColumn("new_column_3", expression_3)
```

In [ ]:
df_final = df.withColumn("discount_amount", col("amount") * 0.10) \
    .withColumn("final_amount", col("amount") - col("discount_amount"))

df_final.show()

## 20. Update Existing Column

If the column already exists, `withColumn()` updates it.

Example:

```text
Increase amount by 100
```


In [ ]:
df_updated = df.withColumn("amount", col("amount") + 100)

df_updated.show()

## 21. Important Difference: Original DataFrame Does Not Change

PySpark DataFrames are immutable.

Simple meaning:

```text
When we apply withColumn(), Spark creates a new DataFrame.
The original DataFrame does not change.
```

Let us check the original `df`.


In [ ]:
df.show()

## 22. Commands Covered in This Notebook

| Command | Purpose |
|---|---|
| `select()` | Select columns |
| `filter()` | Filter rows |
| `&` | AND condition |
| `|` | OR condition |
| `col()` | Refer to a DataFrame column |
| `lit()` | Add fixed value |
| `withColumn()` | Add or update column |


## 23. Stop SparkSession

At the end of practice, stop SparkSession to release resources.


In [ ]:
spark.stop()